In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
sql("SET spark.databricks.delta.formatCheck.enabled=false")

In [0]:
# dbutils.widgets.text("run_date", "")
# run_date = dbutils.widgets.get("run_date")
# print(f"Processing for: {run_date}")
# LOOKBACK_END_DATE = run_date

# print(f"LOOKBACK_END_DATE: {LOOKBACK_END_DATE}")
# LOOKBACK_START_DATE = (datetime.strptime(LOOKBACK_END_DATE, "%Y-%m-%d") - timedelta(days=365)).strftime("%Y-%m-%d")
# CURRENT_WEEK_START_DATE = (datetime.strptime(LOOKBACK_END_DATE, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")
# CURRENT_WEEK_END_DATE = (datetime.strptime(LOOKBACK_END_DATE, "%Y-%m-%d") + timedelta(weeks=1)).strftime("%Y-%m-%d")

In [0]:
LOOKBACK_START_DATE = '2025-07-31'
LOOKBACK_END_DATE = '2026-07-31' # Friday
CURRENT_WEEK_START_DATE = '2026-08-01' # saturday   
CURRENT_WEEK_END_DATE = '2026-08-31' # Friday

In [0]:
LOOKBACK_START_DATE, LOOKBACK_END_DATE, CURRENT_WEEK_START_DATE, CURRENT_WEEK_END_DATE

In [0]:
# Lookback 1 year =>  date > LOOKBACK_END_DATE-1year and date <= LOOKBACK_END_DATE => PD NBRX
# Current Week =>  date > LOOKBACK_END_DATE and date <= CURRENT_WEEK_END_DATE

In [0]:
# Loading all past claim as well as latest refresh claims
query = f'''
            SELECT DISTINCT
                A.CLAIM_ID,
                A.PATIENT_ID,
                TO_DATE(A.SVC_DT) AS SVC_DT,
                A.PROVIDER_ID,
                A.CLAIM_TYPE,
                TRIM(UPPER(B.PRODUCT_GROUP)) AS PRODUCT_GROUP
            FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX A
            INNER JOIN PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD B
            USING (NDC_CD)
            WHERE TO_DATE(A.SVC_DT) > DATE '{LOOKBACK_START_DATE}'
            AND TO_DATE(A.SVC_DT) <= DATE '{CURRENT_WEEK_END_DATE}'
            AND CLAIM_TYPE = 'PD'
        '''
all_claims = Get_Data_Snowflakes(query)
print("Shape :", all_claims.count(), len(all_claims.columns))

In [0]:
# all_claims.display()

In [0]:
# Signifies number of weeks data is refreshed for
# num_weeks_refresh = 13

In [0]:
# # Assumption : Data is available for last 13 weeks (Update ./00_params file in case of any change)
# from datetime import datetime, timedelta
# max_date_dt = datetime.strptime(str(max_weekly_date), "%Y-%m-%d")

# # Compute the date 13 weeks (91 days) before max_date
# thirteen_weeks_earlier = max_date_dt - timedelta(weeks = num_weeks_refresh)

# # Print the calculated date
# print("Max Date:", max_weekly_date)
# print("Date 13 weeks earlier:", thirteen_weeks_earlier.strftime("%Y-%m-%d"))

In [0]:
# # Filter data for lookback 1 year & the latest 13 weeks of new data
# all_claims_last_1_year = all_claims.filter(F.col("SVC_DT") >= date_add(F.lit(thirteen_weeks_earlier), -365))
# # all_claims_last_1_year = all_claims_last_1_year.cache()
# print("Shape :", all_claims_last_1_year.count(), len(all_claims_last_1_year.columns))

In [0]:
# Date Range
data_end = all_claims.agg(max("SVC_DT").alias("MAX_SVC_DT")).collect()[0]["MAX_SVC_DT"]
data_start = all_claims.agg(min("SVC_DT").alias("MIN_SVC_DT")).collect()[0]["MIN_SVC_DT"]
print("Processed data available from", data_start, "to", data_end)

In [0]:
all_claims.select("PROVIDER_ID").distinct().count()

In [0]:
all_claims.select("CLAIM_TYPE").distinct().count()

#### Adding Product related information

In [0]:
!pip install openpyxl

In [0]:
# Loading Product Name to Group mapping
priority_df = pd.read_excel(prod_group_mapping_path)
priority_df = priority_df[['PRODUCT_NAME', 'PRODUCT_GROUP', 'PRIORITY']]
priority_df = spark.createDataFrame(priority_df)
# priority_df =  priority_df.filter(F.col("PRODUCT_GROUP")=="KERENDIA")

In [0]:
priority_df.groupBy("PRODUCT_GROUP").count().display()

In [0]:
# Adding higher level Product Group
all_claims = all_claims.withColumnRenamed("PRODUCT_GROUP","PRODUCT_NAME")
all_claims = all_claims.join(priority_df, on = 'PRODUCT_NAME', how='inner')
all_claims = all_claims.drop("PRODUCT_NAME", "PRIORITY")
all_claims = all_claims.filter(col("PRODUCT_GROUP").isNotNull())
# all_claims = all_claims.cache()
print('Shape :', all_claims.count(), len(all_claims.columns))

In [0]:
all_claims = all_claims.drop("PRIORITY")

In [0]:
all_claims.filter(col('SVC_DT')>LOOKBACK_END_DATE).display()

In [0]:
all_claims.filter(col('SVC_DT')>LOOKBACK_END_DATE).count()

In [0]:
all_claims.filter(col('SVC_DT')>LOOKBACK_END_DATE).groupBy("CLAIM_TYPE").count().display()

In [0]:
all_claims.filter((col('SVC_DT')>LOOKBACK_END_DATE) & (col('PRODUCT_GROUP')=='KERENDIA')).groupBy("CLAIM_TYPE").count().display()

In [0]:
all_claims.filter((col('SVC_DT')>LOOKBACK_END_DATE) & (col('PRODUCT_GROUP')=='KERENDIA')).count()

In [0]:
all_claims.filter((col('SVC_DT')>LOOKBACK_END_DATE) & (col('PRODUCT_GROUP')=='KERENDIA')).select('PATIENT_ID').distinct().count()

## LOOKBACK

In [0]:
all_lookback_claims = all_claims.filter((col('SVC_DT')<=LOOKBACK_END_DATE))

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, min as spark_min, lag, expr, when, row_number

# Step 1: Assign rank per patient-product-date to pick the first claim
dedup_window = Window.partitionBy("PATIENT_ID", "PRODUCT_GROUP","SVC_DT").orderBy("CLAIM_ID")

all_lookback_claims = all_lookback_claims.withColumn("CLAIM_RANK", row_number().over(dedup_window))

# Step 2: Define a window for NBRx calculation (per patient-product history)
window_spec = Window.partitionBy("PATIENT_ID", "PRODUCT_GROUP").orderBy("SVC_DT")

# Compute the first-ever claim date per patient-product
all_lookback_claims = all_lookback_claims.withColumn("MIN_SVC_DT", spark_min("SVC_DT").over(window_spec))

# Compute the lagged service date (previous claim date for that product)
all_lookback_claims = all_lookback_claims.withColumn("LAG_SVC_DT", lag("SVC_DT").over(window_spec))

# Step 3: Calculate NBRX_FLAG only for the first claim per day
all_lookback_claims_w_nbrx = all_lookback_claims.withColumn(
    "NBRX_FLAG",
    when((col("MIN_SVC_DT") == col("SVC_DT")) & (col("CLAIM_RANK") == 1), 1)
    .when((col("LAG_SVC_DT").isNotNull()) & (col("LAG_SVC_DT") < expr("date_add(SVC_DT, -365)")) & (col("CLAIM_RANK") == 1), 1)
    .otherwise(0)
)

# Step 4: Drop intermediate columns
all_lookback_claims_w_nbrx = all_lookback_claims_w_nbrx.drop("MIN_SVC_DT", "LAG_SVC_DT", "CLAIM_RANK")

# Cache the resulting DataFrame
# all_lookback_claims_w_nbrx = all_lookback_claims_w_nbrx.cache()
print('Shape :', all_lookback_claims_w_nbrx.count(), len(all_lookback_claims_w_nbrx.columns))

In [0]:
# ── Paid (PD) NBRX Claims ──────────────────────────────────────────────

from pyspark.sql.functions import sum as spark_sum, col


hcp_lookback_kerendia_paid_nbrx_rollup = (
    all_lookback_claims_w_nbrx
    .groupBy("PROVIDER_ID")
    .agg(
        F.sum(
            F.when((F.col("PRODUCT_GROUP") == "KERENDIA") & (col("NBRX_FLAG") == 1) & (col("CLAIM_TYPE") == "PD"), F.col("NBRX_FLAG")).otherwise(0)
        ).alias("NBRX_KERENDIA_LOOKBACK_PD_CLAIMS")
    )
)

# Fill nulls in NBRX_KERENDIA_LOOKBACK_PD_CLAIMS with 0
hcp_lookback_kerendia_paid_nbrx_rollup = hcp_lookback_kerendia_paid_nbrx_rollup.fillna({"NBRX_KERENDIA_LOOKBACK_PD_CLAIMS": 0})

In [0]:
hcp_lookback_kerendia_paid_nbrx_rollup.select('PROVIDER_ID').distinct().count()

In [0]:
hcp_lookback_kerendia_paid_nbrx_rollup.filter(col("NBRX_KERENDIA_LOOKBACK_PD_CLAIMS") == 0).select('PROVIDER_ID').distinct().count()

In [0]:
hcp_lookback_kerendia_paid_nbrx_rollup.write.mode("overwrite").option("overwriteSchema", "true").parquet("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/speaker_program_dev/2026_08_01/RAW_LAAD_NBRX_EW_NW/lookback_nbrx_data.parquet")

## Current week NBRx

In [0]:
all_curr_week_claims = all_claims.filter(col('PRODUCT_GROUP')=='KERENDIA')

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, min as spark_min, lag, expr, when, row_number

# Step 1: Assign rank per patient-product-date to pick the first claim
dedup_window = Window.partitionBy("PATIENT_ID", "PRODUCT_GROUP","SVC_DT").orderBy("CLAIM_ID")

all_curr_week_claims = all_curr_week_claims.withColumn("CLAIM_RANK", row_number().over(dedup_window))

# Step 2: Define a window for NBRx calculation (per patient-product history)
window_spec = Window.partitionBy("PATIENT_ID", "PRODUCT_GROUP").orderBy("SVC_DT")

# Compute the first-ever claim date per patient-product
all_curr_week_claims = all_curr_week_claims.withColumn("MIN_SVC_DT", spark_min("SVC_DT").over(window_spec))

# Compute the lagged service date (previous claim date for that product)
all_curr_week_claims = all_curr_week_claims.withColumn("LAG_SVC_DT", lag("SVC_DT").over(window_spec))

# Step 3: Calculate NBRX_FLAG only for the first claim per day
all_curr_week_claims_w_nbrx = all_curr_week_claims.withColumn(
    "NBRX_FLAG",
    when((col("MIN_SVC_DT") == col("SVC_DT")) & (col("CLAIM_RANK") == 1), 1)
    .when((col("LAG_SVC_DT").isNotNull()) & (col("LAG_SVC_DT") < expr("date_add(SVC_DT, -365)")) & (col("CLAIM_RANK") == 1), 1)
    .otherwise(0)
)

# Step 4: Drop intermediate columns
all_curr_week_claims_w_nbrx = all_curr_week_claims_w_nbrx.drop("MIN_SVC_DT", "LAG_SVC_DT", "CLAIM_RANK")

# Cache the resulting DataFrame
# all_curr_week_claims_w_nbrx = all_curr_week_claims_w_nbrx.cache()
print('Shape :', all_curr_week_claims_w_nbrx.count(), len(all_curr_week_claims_w_nbrx.columns))

In [0]:
all_curr_week_claims_w_nbrx = all_curr_week_claims_w_nbrx.filter((col("SVC_DT") >= CURRENT_WEEK_START_DATE) & (col("SVC_DT") <= CURRENT_WEEK_END_DATE))

In [0]:
from pyspark.sql.functions import sum as spark_sum

# Step 1: Drop Claim granularity columns (CLAIM_ID, SVC_DT)
hcp_curr_week_nbrx_overall = all_curr_week_claims_w_nbrx.select(
    "SVC_DT", "PATIENT_ID", "PRODUCT_GROUP", "PROVIDER_ID", "CLAIM_TYPE", "NBRX_FLAG"
)

hcp_curr_week_nbrx_overall_rollup = (
    hcp_curr_week_nbrx_overall
    .filter(F.col("PRODUCT_GROUP") == "KERENDIA")
    .groupBy("PROVIDER_ID")
    .agg(
        F.sum("NBRX_FLAG").alias("NBRX_KERENDIA_CURR_WEEK_OVERALL_CLAIMS"),
        F.sum(F.when(F.col("CLAIM_TYPE") == "PD", F.col("NBRX_FLAG")).otherwise(0)).alias("NBRX_KERENDIA_CURR_WEEK_PD_CLAIMS"),
        F.sum(F.when(F.col("CLAIM_TYPE") == "RJ", F.col("NBRX_FLAG")).otherwise(0)).alias("NBRX_KERENDIA_CURR_WEEK_RJ_CLAIMS"),
        F.sum(F.when(F.col("CLAIM_TYPE") == "RV", F.col("NBRX_FLAG")).otherwise(0)).alias("NBRX_KERENDIA_CURR_WEEK_RV_CLAIMS")
    )
)

In [0]:
hcp_curr_week_nbrx_overall_rollup.select('PROVIDER_ID').distinct().count()

In [0]:
hcp_curr_week_nbrx_overall_rollup.filter(col("NBRX_KERENDIA_CURR_WEEK_OVERALL_CLAIMS") > 0).select('PROVIDER_ID').distinct().count()

In [0]:
all_curr_week_claims_w_nbrx.select('SVC_DT').distinct().display()

In [0]:
hcp_curr_week_nbrx_overall_rollup.write.mode("overwrite").option("overwriteSchema", "true").parquet("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/speaker_program_dev/2026_08_01/RAW_LAAD_NBRX_EW_NW/prediction_nbrx_data.parquet")

In [0]:
# hcp_curr_week_nbrx_overall_rollup = spark.read.parquet(f"/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/weekly_alerts/{weekly_run_date}_run/artifacts/current_week_hcp_rollup_{LOOKBACK_END_DATE}.parquet")

In [0]:
# hcp_lookback_kerendia_paid_nbrx_rollup = spark.read.parquet(f"/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/weekly_alerts/{weekly_run_date}_run/artifacts/lookback_window_hcp_rollup_{LOOKBACK_END_DATE}.parquet")

EW

In [0]:
EW_writers = hcp_curr_week_nbrx_overall_rollup.filter(col("NBRX_KERENDIA_CURR_WEEK_PD_CLAIMS") > 0).join(hcp_lookback_kerendia_paid_nbrx_rollup.filter(col("NBRX_KERENDIA_LOOKBACK_PD_CLAIMS") > 0), on = 'PROVIDER_ID', how='inner')
print(EW_writers.count())
EW_writers.display()

In [0]:
from pyspark.sql.functions import sum as spark_sum

EW_writers.select(
    spark_sum("NBRX_KERENDIA_CURR_WEEK_OVERALL_CLAIMS").alias("SUM_OVERALL_NBRX"),
    spark_sum("NBRX_KERENDIA_CURR_WEEK_PD_CLAIMS").alias("SUM_PD_NBRX"),
    spark_sum("NBRX_KERENDIA_CURR_WEEK_RV_CLAIMS").alias("SUM_RV_NBRX"),
    spark_sum("NBRX_KERENDIA_CURR_WEEK_RJ_CLAIMS").alias("SUM_RJ_NBRX")
).display()

NW Writers

In [0]:
NW_writers = hcp_curr_week_nbrx_overall_rollup.filter(col("NBRX_KERENDIA_CURR_WEEK_PD_CLAIMS") > 0).join(hcp_lookback_kerendia_paid_nbrx_rollup.filter(col("NBRX_KERENDIA_LOOKBACK_PD_CLAIMS") > 0).select("PROVIDER_ID").distinct(), on = 'PROVIDER_ID', how='left_anti')
print(NW_writers.count())
NW_writers.display()

In [0]:
# NW_writers.write.mode("overwrite").option("overwriteSchema", "true").parquet(f"/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/weekly_alerts/{weekly_run_date}_run/NW_writer/NW_writer_list_{LOOKBACK_END_DATE}.parquet")

In [0]:
from pyspark.sql.functions import sum as spark_sum

NW_writers.select(
    spark_sum("NBRX_KERENDIA_CURR_WEEK_OVERALL_CLAIMS").alias("SUM_OVERALL_NBRX"),
    spark_sum("NBRX_KERENDIA_CURR_WEEK_PD_CLAIMS").alias("SUM_PD_NBRX"),
    spark_sum("NBRX_KERENDIA_CURR_WEEK_RV_CLAIMS").alias("SUM_RV_NBRX"),
    spark_sum("NBRX_KERENDIA_CURR_WEEK_RJ_CLAIMS").alias("SUM_RJ_NBRX")
).display()

In [0]:
# all_curr_week_claims_w_nbrx.filter(F.col("PROVIDER_ID").isin([row['PROVIDER_ID'] for row in overlap.select('PROVIDER_ID').distinct().collect()])).display()

In [0]:
# 29584447



In [0]:
# all_curr_week_claims_w_nbrx.filter(F.col("PATIENT_ID") == "29584447").display()

In [0]:
# all_curr_week_claims_w_nbrx.filter(F.col("PROVIDER_ID").isin([row['PROVIDER_ID'] for row in overlap.select('PROVIDER_ID').distinct().collect()])).display()